# USDA-Net — Subject-Independent 3-Class Motor-Imagery EEG

## Selected dataset: PhysioNet EEGMMIDB

**Research objective:** strict subject-independent motor-imagery classification using a 3-class task:

- **Left hand**
- **Right hand**
- **Feet**

This notebook implements the proposed **USDA-Net (Universal Subject-Domain Adaptive Network)** architecture without replacing it with EEGNet, a plain Transformer, or another model.

### Source-of-truth experimental contract

The supplied project modules freeze the following preprocessing and evaluation interface:

- common EEG sensor space: **22 channels**;
- target sampling rate: **160 Hz**;
- primary bandpass: **8–30 Hz**;
- epoch: **0–4 s after the task cue**;
- epoch shape: **(22, 640)**;
- continuous preprocessing before epoch extraction;
- average reference over the 22 retained EEG channels;
- no interpolation;
- preprocessing cache remains **unnormalized**;
- normalization is fit from **source training subjects only**;
- strict unseen-subject LOSO is the primary evaluation;
- target-subject statistics are forbidden before final testing.

The label/run semantics are also frozen from the supplied label-harmonization module: EEGMMIDB R04/R08/R12 use T1=left-fist imagery and T2=right-fist imagery; R06/R10/R14 use T2=both-feet imagery, while T1=both-fists imagery is excluded.

## Important scientific rule

The notebook does **not** assume that 80% accuracy is guaranteed. The target is:

\[
\text{mean strict-LOSO accuracy} \ge 80\%
\]

but the result is accepted only when it is actually obtained under the frozen leakage policy.

The notebook reports accuracy, balanced accuracy, macro-F1, Cohen's kappa, per-class recall, fold mean ± SD, confidence intervals, and the confusion matrix.

In [7]:
# CELL 1 — INSTALL / IMPORTS / GLOBAL CONFIGURATION
# Run the pip line only if your environment does not already have these packages.
# !pip install mne h5py scipy scikit-learn pandas numpy matplotlib seaborn tqdm

from __future__ import annotations

import os
import re
import gc
import json
import math
import time
import random
import hashlib
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from collections import Counter

import numpy as np
import pandas as pd
import h5py
import mne
import matplotlib.pyplot as plt

from scipy import signal
from scipy.stats import t as student_t
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    cohen_kappa_score,
    confusion_matrix,
    classification_report,
)
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

warnings.filterwarnings("ignore", category=RuntimeWarning)
mne.set_log_level("ERROR")

SEED = 20260822

def seed_everything(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything()

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print("PyTorch:", torch.__version__)
print("MNE    :", mne.__version__)
print("Device :", DEVICE)
print("Seed   :", SEED)

PyTorch: 2.10.0
MNE    : 1.11.0
Device : mps
Seed   : 20260822


In [8]:
# CELL 2 — PROJECT PATHS
# Change ONLY PROJECT_ROOT / EEGMMIDB_ROOT if your local directory differs.

PROJECT_ROOT = Path("/Users/ashokvarmabevara/Project2")
EEGMMIDB_ROOT = PROJECT_ROOT / "eegmmidb"

OUTPUT_ROOT = PROJECT_ROOT / "cross_dataset_mi_project"
CACHE_ROOT = OUTPUT_ROOT / "cache"
RESULTS_ROOT = OUTPUT_ROOT / "results" / "USDA_Net_PhysioNet_3Class"
FIGURES_ROOT = OUTPUT_ROOT / "figures" / "USDA_Net_PhysioNet_3Class"
CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints" / "USDA_Net_PhysioNet_3Class"
MANIFEST_ROOT = OUTPUT_ROOT / "manifests"

for p in [OUTPUT_ROOT, CACHE_ROOT, RESULTS_ROOT, FIGURES_ROOT, CHECKPOINT_ROOT, MANIFEST_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

# Existing validated project cache from Module 5 v2. If present, this notebook
# can use it directly and subset EEGMMIDB without rebuilding raw EDFs.
VALIDATED_CACHE_CANDIDATES = [
    CACHE_ROOT / "module_5_v2_preprocessed_epochs_160hz_8_30hz_continuous.h5",
    CACHE_ROOT / "module_5_preprocessed_epochs_160hz_8_30hz.h5",
]

EXISTING_CACHE = next((p for p in VALIDATED_CACHE_CANDIDATES if p.exists()), None)

print("Project root :", PROJECT_ROOT)
print("EEGMMIDB     :", EEGMMIDB_ROOT)
print("Existing cache:", EXISTING_CACHE)

Project root : /Users/ashokvarmabevara/Project2
EEGMMIDB     : /Users/ashokvarmabevara/Project2/eegmmidb
Existing cache: /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/cache/module_5_v2_preprocessed_epochs_160hz_8_30hz_continuous.h5


In [9]:
# CELL 3 — FROZEN DATASET / TASK SPECIFICATION

DATASET = "EEGMMIDB"
PRIMARY_CLASSES = ["left", "right", "feet"]
CLASS_TO_ID = {name: i for i, name in enumerate(PRIMARY_CLASSES)}
ID_TO_CLASS = {i: name for name, i in CLASS_TO_ID.items()}

TARGET_SFREQ = 160.0
LOW_HZ, HIGH_HZ = 8.0, 30.0
TMIN, TMAX = 0.0, 4.0
N_CHANNELS = 22
N_SAMPLES = 640

# Frozen common channel order from Module 4.
FROZEN_COMMON_CHANNELS = [
    "Fz", "FC3", "FC1", "FCz", "FC2", "FC4",
    "C5", "C3", "C1", "Cz", "C2", "C4", "C6",
    "CP3", "CP1", "CPz", "CP2", "CP4",
    "P1", "Pz", "P2", "POz",
]

assert len(FROZEN_COMMON_CHANNELS) == 22
assert N_SAMPLES == int((TMAX - TMIN) * TARGET_SFREQ)

# EEGMMIDB run-dependent label semantics frozen by Module 3.
UNILATERAL_RUNS = {"R04", "R08", "R12"}
BILATERAL_RUNS = {"R06", "R10", "R14"}
PRIMARY_IMAGERY_RUNS = UNILATERAL_RUNS | BILATERAL_RUNS

EEGMMIDB_RUN_MAPPING = {}
for run in UNILATERAL_RUNS:
    EEGMMIDB_RUN_MAPPING[run] = {
        "T0": (None, False, "rest"),
        "T1": ("left", True, "left fist imagery"),
        "T2": ("right", True, "right fist imagery"),
    }
for run in BILATERAL_RUNS:
    EEGMMIDB_RUN_MAPPING[run] = {
        "T0": (None, False, "rest"),
        "T1": (None, False, "both fists imagery"),
        "T2": ("feet", True, "both feet imagery"),
    }

print("Task classes:", PRIMARY_CLASSES)
print("Input shape :", (N_CHANNELS, N_SAMPLES))
print("Sampling    :", TARGET_SFREQ, "Hz")
print("Bandpass    :", (LOW_HZ, HIGH_HZ), "Hz")
print("Epoch       :", (TMIN, TMAX), "s")
print("EEGMMIDB MI runs:", sorted(PRIMARY_IMAGERY_RUNS))

Task classes: ['left', 'right', 'feet']
Input shape : (22, 640)
Sampling    : 160.0 Hz
Bandpass    : (8.0, 30.0) Hz
Epoch       : (0.0, 4.0) s
EEGMMIDB MI runs: ['R04', 'R06', 'R08', 'R10', 'R12', 'R14']


## Architecture — fixed USDA-Net design

The model remains the architecture defined for this project:

```text
RAW EEG [B,C,T]
      │
      ├────────────── Multi-Scale Temporal CNN ────────┐
      │                                                 │
      └────────────── Learnable Spectral Filterbank ────┤
                                                        ▼
                                                 Feature Fusion
                                                        │
                                                 Spatial Encoder
                                                        │
                                                 Channel Attention
                                                        │
                                                  Residual TCN
                                                        │
                                               Transformer × 3
                                                        │
                                               Attention Pooling
                                                        │
                                                 128-D Embedding
                                                        │
                           ┌────────────────────────────┼───────────────────┐
                           │                            │                   │
                       Class Head                  DANN Domain        Class Prototypes
                           │                            │                   │
                     Left/Right/Feet                Subject ID          Prototype Loss
                           │                                                │
                           └──────────────────────── SupCon ─────────────────┘
```

The training objective is:

\[
L_{total}=L_{CE}+\lambda_dL_{DANN}+\lambda_pL_{prototype}+\lambda_cL_{SupCon}
\]

The domain branch uses gradient reversal so the shared representation becomes less subject-specific.

In [10]:
# CELL 4 — CACHE DISCOVERY AND RAW-DATA FALLBACK

RAW_CACHE_PATH = CACHE_ROOT / "USDA_Net_EEGMMIDB_3class_raw_preprocessed.h5"

# Main data source selection:
# 1) Prefer the already validated Module 5 v2 cache when present.
# 2) Otherwise use this notebook's raw-EDF builder.
ACTIVE_CACHE = EXISTING_CACHE if EXISTING_CACHE is not None else RAW_CACHE_PATH

print("Active cache candidate:", ACTIVE_CACHE)

if ACTIVE_CACHE.exists():
    print("Using existing preprocessed cache.")
else:
    print("No preprocessed cache found.")
    print("The raw EEGMMIDB builder cells below will construct it from local EDF files.")

Active cache candidate: /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/cache/module_5_v2_preprocessed_epochs_160hz_8_30hz_continuous.h5
Using existing preprocessed cache.


In [11]:
# CELL 5 — GENERAL CHANNEL-NAME NORMALIZATION

def physionet_standard_name(raw_name: str) -> str:
    s = str(raw_name).strip().upper()
    s = re.sub(r"[.\s]+$", "", s)
    s = s.replace("-", "").replace("_", "")
    aliases = {
        "FP1": "Fp1", "FP2": "Fp2", "FPZ": "Fpz",
        "FZ": "Fz", "FCZ": "FCz", "CZ": "Cz",
        "CPZ": "CPz", "PZ": "Pz", "POZ": "POz",
        "OZ": "Oz", "IZ": "Iz", "AFZ": "AFz",
    }
    if s in aliases:
        return aliases[s]
    m = re.fullmatch(r"([A-Z]+)(\d+)", s)
    if m:
        prefix, number = m.groups()
        if prefix in {"F", "C", "P", "O", "FC", "CP", "AF", "PO", "FT", "TP"}:
            return f"{prefix}{number}"
    return s

# The BCI-2a common montage is intentionally retained even though this notebook
# trains on EEGMMIDB alone; this is the frozen common sensor space from Module 4.
print(pd.DataFrame({"common_order": np.arange(N_CHANNELS), "channel": FROZEN_COMMON_CHANNELS}).to_string(index=False))

 common_order channel
            0      Fz
            1     FC3
            2     FC1
            3     FCz
            4     FC2
            5     FC4
            6      C5
            7      C3
            8      C1
            9      Cz
           10      C2
           11      C4
           12      C6
           13     CP3
           14     CP1
           15     CPz
           16     CP2
           17     CP4
           18      P1
           19      Pz
           20      P2
           21     POz


In [12]:
# CELL 6 — RAW EEGMMIDB DISCOVERY

def discover_physionet_edf(root: Path) -> pd.DataFrame:
    rows = []
    for p in sorted(root.rglob("*.edf")):
        m = re.search(r"S(\d{3})R(\d{2})$", p.stem.upper())
        if not m:
            continue
        subject, run_num = m.groups()
        rows.append({
            "dataset": "EEGMMIDB",
            "subject": f"S{subject}",
            "run": f"R{run_num}",
            "recording_id": p.stem.upper(),
            "filename": p.name,
            "absolute_path": str(p.resolve()),
        })
    return pd.DataFrame(rows)

edf_df = discover_physionet_edf(EEGMMIDB_ROOT)

if len(edf_df):
    print("EDF files discovered:", len(edf_df))
    print("Subjects:", edf_df.subject.nunique())
    print("Subjects sample:", sorted(edf_df.subject.unique())[:10])
    print("Runs:", sorted(edf_df.run.unique()))
    print(edf_df.groupby(["subject", "run"]).size().head(20))
else:
    print("No EDF files were discovered under:", EEGMMIDB_ROOT)

EDF files discovered: 1526
Subjects: 109
Subjects sample: ['S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S007', 'S008', 'S009', 'S010']
Runs: ['R01', 'R02', 'R03', 'R04', 'R05', 'R06', 'R07', 'R08', 'R09', 'R10', 'R11', 'R12', 'R13', 'R14']
subject  run
S001     R01    1
         R02    1
         R03    1
         R04    1
         R05    1
         R06    1
         R07    1
         R08    1
         R09    1
         R10    1
         R11    1
         R12    1
         R13    1
         R14    1
S002     R01    1
         R02    1
         R03    1
         R04    1
         R05    1
         R06    1
dtype: int64


## Raw-cache builder

This section is only executed when the validated project cache is not available. It follows the supplied modules rather than introducing a new preprocessing policy:

1. keep only EEGMMIDB imagery runs R04/R08/R12/R06/R10/R14;
2. interpret T1/T2 according to run semantics;
3. retain only Left / Right / Feet events;
4. resample the continuous signal to 160 Hz using `scipy.signal.resample_poly`;
5. retain the frozen 22-channel common montage with **no interpolation**;
6. apply average reference and the frozen 8–30 Hz FIR filter on the continuous signal;
7. extract 4-second post-cue epochs;
8. write the cache **without normalization**.

In [13]:
# CELL 7 — RAW-CACHE EVENT / PREPROCESSING HELPERS

def annotation_label(desc) -> str:
    s = str(desc).strip().upper()
    # PhysioNet annotations are commonly T0/T1/T2; tolerate descriptions such as 'T1'.
    m = re.search(r"T[012]", s)
    return m.group(0) if m else s


def resample_continuous(X: np.ndarray, sfreq: float, target: float) -> np.ndarray:
    if abs(float(sfreq) - float(target)) < 1e-9:
        return X.astype(np.float32, copy=False)
    ratio = target / float(sfreq)
    # Rational approximation for scipy.signal.resample_poly.
    from fractions import Fraction
    frac = Fraction(ratio).limit_denominator(1000)
    return signal.resample_poly(X, frac.numerator, frac.denominator, axis=-1).astype(np.float32)


def exact_common_channel_indices(raw_names: List[str]) -> List[int]:
    normalized = [physionet_standard_name(n) for n in raw_names]
    idx = []
    for ch in FROZEN_COMMON_CHANNELS:
        matches = [i for i, n in enumerate(normalized) if n == ch]
        if len(matches) != 1:
            raise RuntimeError(f"Expected exactly one PhysioNet channel for {ch}; found {matches} from {raw_names}")
        idx.append(matches[0])
    return idx


def preprocess_raw_eeg_file(path: Path) -> Tuple[np.ndarray, float, List[dict]]:
    raw = mne.io.read_raw_edf(str(path), preload=True, verbose="ERROR")
    try:
        sf = float(raw.info["sfreq"])
        picks = exact_common_channel_indices(raw.ch_names)
        X = raw.get_data(picks=picks).astype(np.float64)

        # Resample the continuous signal before filtering and epoching.
        X = resample_continuous(X, sf, TARGET_SFREQ).astype(np.float64)

        # Continuous average reference.
        X = X - np.mean(X, axis=0, keepdims=True)

        # Frozen 8–30 Hz continuous FIR bandpass.
        info = mne.create_info(FROZEN_COMMON_CHANNELS, sfreq=TARGET_SFREQ, ch_types="eeg")
        tmp = mne.io.RawArray(X, info, verbose="ERROR")
        tmp.filter(
            l_freq=LOW_HZ,
            h_freq=HIGH_HZ,
            method="fir",
            phase="zero",
            fir_design="firwin",
            verbose="ERROR",
        )
        X = tmp.get_data().astype(np.float32)

        events = []
        for onset, duration, desc in zip(raw.annotations.onset, raw.annotations.duration, raw.annotations.description):
            token = annotation_label(desc)
            onset_sec = float(onset)
            events.append({"onset_sec": onset_sec, "token": token, "duration_sec": float(duration), "description": str(desc)})
        return X, sf, events
    finally:
        raw.close()


def extract_retained_epochs_from_recording(
    X: np.ndarray,
    events: List[dict],
    subject: str,
    run: str,
    recording_id: str,
    source_sfreq: float,
):
    if run not in EEGMMIDB_RUN_MAPPING:
        return []
    mapping = EEGMMIDB_RUN_MAPPING[run]
    epochs = []
    n_expected = int(round((TMAX - TMIN) * TARGET_SFREQ))
    for event_index, e in enumerate(events):
        token = e["token"]
        if token not in mapping:
            continue
        cls, keep, meaning = mapping[token]
        if not keep:
            continue
        start = int(round((e["onset_sec"] + TMIN) * TARGET_SFREQ))
        stop = start + n_expected
        if start < 0 or stop > X.shape[-1]:
            continue
        epoch = X[:, start:stop]
        if epoch.shape != (N_CHANNELS, N_SAMPLES):
            continue
        if not np.isfinite(epoch).all():
            continue
        epochs.append({
            "X": epoch.astype(np.float32),
            "subject": subject,
            "run": run,
            "recording_id": recording_id,
            "event_index": event_index,
            "raw_event": token,
            "raw_meaning": meaning,
            "harmonized_class": cls,
            "source_sfreq_hz": float(source_sfreq),
        })
    return epochs

In [14]:
# CELL 8 — BUILD EEGMMIDB RAW CACHE WHEN NEEDED

if not ACTIVE_CACHE.exists():
    assert len(edf_df) > 0, "No EEGMMIDB EDF files found. Set EEGMMIDB_ROOT correctly in CELL 2."
    candidate = edf_df[edf_df["run"].isin(PRIMARY_IMAGERY_RUNS)].copy()
    print("Candidate imagery recordings:", len(candidate))

    all_epochs = []
    failures = []
    sampling_rows = []

    for _, row in tqdm(candidate.iterrows(), total=len(candidate), desc="Preprocessing EEGMMIDB"):
        try:
            Xc, sf, ev = preprocess_raw_eeg_file(Path(row["absolute_path"]))
            sampling_rows.append({"subject": row.subject, "run": row.run, "recording_id": row.recording_id, "source_sfreq_hz": sf})
            all_epochs.extend(
                extract_retained_epochs_from_recording(
                    Xc, ev, row.subject, row.run, row.recording_id, sf
                )
            )
        except Exception as exc:
            failures.append({"recording_id": row.recording_id, "subject": row.subject, "run": row.run, "error": repr(exc)})

    print("Retained epochs:", len(all_epochs))
    print("Failures      :", len(failures))
    if failures:
        display(pd.DataFrame(failures).head(20))

    if not all_epochs:
        raise RuntimeError("No epochs were created. Inspect EDF discovery, channel labels, and annotations.")

    X_all = np.stack([r["X"] for r in all_epochs]).astype(np.float32)
    meta = pd.DataFrame([{k:v for k,v in r.items() if k != "X"} for r in all_epochs])

    assert X_all.shape[1:] == (22, 640)
    assert np.isfinite(X_all).all()
    assert set(meta.harmonized_class.unique()) == set(PRIMARY_CLASSES)

    if RAW_CACHE_PATH.exists():
        RAW_CACHE_PATH.unlink()
    with h5py.File(RAW_CACHE_PATH, "w") as h5:
        h5.create_dataset("X", data=X_all, compression="gzip", compression_opts=4, chunks=(min(32, len(X_all)), 22, 640))
        mgrp = h5.create_group("metadata")
        for col in meta.columns:
            vals = meta[col].astype(str).to_numpy(dtype="S64")
            mgrp.create_dataset(col, data=vals)
        h5.attrs["normalized"] = False
        h5.attrs["target_sfreq_hz"] = TARGET_SFREQ
        h5.attrs["bandpass_low_hz"] = LOW_HZ
        h5.attrs["bandpass_high_hz"] = HIGH_HZ
        h5.attrs["epoch_tmin_sec"] = TMIN
        h5.attrs["epoch_tmax_sec"] = TMAX
        h5.attrs["n_channels"] = N_CHANNELS
        h5.attrs["n_samples"] = N_SAMPLES
        h5.attrs["reference"] = "average reference over frozen 22 common channels"
        h5.attrs["resampling"] = "scipy.signal.resample_poly"
        h5.attrs["filter"] = "MNE FIR zero-phase continuous 8-30 Hz"
    ACTIVE_CACHE = RAW_CACHE_PATH
else:
    print("Raw preprocessing skipped; using:", ACTIVE_CACHE)

Raw preprocessing skipped; using: /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/cache/module_5_v2_preprocessed_epochs_160hz_8_30hz_continuous.h5


In [15]:
# CELL 9 — HDF5 LOADER + EEGMMIDB-ONLY SUBSETTING

class H5Store:
    def __init__(self, path: Path):
        self.path = Path(path)
        self.h5 = None
    def __enter__(self):
        self.h5 = h5py.File(self.path, "r")
        return self
    def __exit__(self, exc_type, exc, tb):
        if self.h5 is not None:
            self.h5.close()
            self.h5 = None

with H5Store(ACTIVE_CACHE) as store:
    X_shape = tuple(store.h5["X"].shape)
    meta = {}
    for key in store.h5["metadata"].keys():
        arr = store.h5["metadata"][key][:]
        meta[key] = [x.decode("utf-8") if isinstance(x, (bytes, np.bytes_)) else str(x) for x in arr]
    meta_df = pd.DataFrame(meta)
    cache_attrs = dict(store.h5.attrs)

print("Cache shape:", X_shape)
print("Metadata columns:", meta_df.columns.tolist())

# The project cache may contain both BCI-IV-2a and EEGMMIDB.
# This project selects EEGMMIDB only.
if "dataset" in meta_df.columns:
    eeg_meta_mask = meta_df["dataset"].astype(str).str.upper().eq("EEGMMIDB")
    if eeg_meta_mask.any():
        eeg_indices = np.flatnonzero(eeg_meta_mask.to_numpy())
    else:
        # Raw cache created above contains only EEGMMIDB.
        eeg_indices = np.arange(len(meta_df))
else:
    eeg_indices = np.arange(len(meta_df))

meta_df = meta_df.iloc[eeg_indices].reset_index(drop=True)
cache_indices = eeg_indices.astype(np.int64)

print("EEGMMIDB epochs:", len(meta_df))
print("EEGMMIDB subjects:", meta_df.subject.nunique())
print("Class distribution:")
print(meta_df["harmonized_class"].value_counts().reindex(PRIMARY_CLASSES))

assert set(meta_df["harmonized_class"].unique()) <= set(PRIMARY_CLASSES)
assert len(meta_df.subject.unique()) >= 2

Cache shape: (9316, 22, 640)
Metadata columns: ['absolute_path', 'dataset', 'event_index', 'filename', 'harmonized_class', 'onset_sec', 'recording_id', 'run', 'source_sfreq_hz', 'subject']
EEGMMIDB epochs: 7372
EEGMMIDB subjects: 109
Class distribution:
harmonized_class
left     2479
right    2438
feet     2455
Name: count, dtype: int64


In [16]:
# CELL 10 — CACHE QA GATE

qa = {}
qa["shape_22x640"] = X_shape[1:] == (22, 640)
qa["finite"] = True
qa["three_classes"] = set(meta_df.harmonized_class.unique()) == set(PRIMARY_CLASSES)
qa["metadata_subject"] = "subject" in meta_df.columns
qa["normalized_false"] = bool(cache_attrs.get("normalized", False)) is False
qa["160_hz"] = float(cache_attrs.get("target_sfreq_hz", TARGET_SFREQ)) == TARGET_SFREQ
qa["8_30_hz"] = float(cache_attrs.get("bandpass_low_hz", LOW_HZ)) == LOW_HZ and float(cache_attrs.get("bandpass_high_hz", HIGH_HZ)) == HIGH_HZ
qa["640_samples"] = int(cache_attrs.get("n_samples", N_SAMPLES)) == N_SAMPLES

# Chunked finite-value scan only on selected EEGMMIDB rows.
with H5Store(ACTIVE_CACHE) as store:
    Xds = store.h5["X"]
    finite_ok = True
    for inds in np.array_split(cache_indices, max(1, math.ceil(len(cache_indices)/64))):
        if len(inds) == 0:
            continue
        block = np.asarray(Xds[inds], dtype=np.float32)
        if not np.isfinite(block).all():
            finite_ok = False
            break
qa["finite"] = finite_ok

print(json.dumps(qa, indent=2))
assert all(qa.values()), "Cache QA failed; do not start training."
print("CACHE QA: PASS")

{
  "shape_22x640": true,
  "finite": true,
  "three_classes": true,
  "metadata_subject": true,
  "normalized_false": true,
  "160_hz": true,
  "8_30_hz": true,
  "640_samples": true
}
CACHE QA: PASS


In [17]:
# CELL 11 — SOURCE-ONLY ROBUST NORMALIZER

class SourceOnlyRobustNormalizer:
    def __init__(self, eps: float = 1e-6):
        self.eps = eps
        self.median_ = None
        self.iqr_ = None
        self.fitted_subjects_ = tuple()

    def fit(self, X: np.ndarray, subjects: List[str]):
        X = np.asarray(X, dtype=np.float32)
        if X.ndim != 3:
            raise ValueError(X.shape)
        vals = X.transpose(1,0,2).reshape(X.shape[1], -1).astype(np.float64)
        self.median_ = np.median(vals, axis=1).astype(np.float32)
        q25 = np.percentile(vals, 25, axis=1)
        q75 = np.percentile(vals, 75, axis=1)
        self.iqr_ = np.maximum(q75-q25, self.eps).astype(np.float32)
        self.fitted_subjects_ = tuple(sorted(set(map(str, subjects))))
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        if self.median_ is None:
            raise RuntimeError("Normalizer is not fitted.")
        return ((np.asarray(X, dtype=np.float32) - self.median_[None,:,None]) /
                self.iqr_[None,:,None]).astype(np.float32)

    def assert_target_excluded(self, target_subject: str):
        if str(target_subject) in set(self.fitted_subjects_):
            raise AssertionError(f"LEAKAGE: target subject {target_subject} entered normalizer fit.")
        return True

print("Source-only normalizer defined.")

Source-only normalizer defined.


## USDA-Net implementation

### Fixed architectural choices

- Multi-scale temporal kernels: **3, 7, 15, 31**
- Learnable spectral bands: **6 bands**, initialized to the project-design ranges
- Fused feature width: **128**
- Depthwise spatial encoder
- Channel attention
- Residual temporal blocks
- **3 Transformer encoder layers**, 4 heads
- Attention pooling
- **128-dimensional embedding**
- Classification head for Left / Right / Feet
- Gradient-reversal subject/domain classifier
- Learnable class-prototype bank

In [28]:
# CELL 12 — GRADIENT REVERSAL + LEARNABLE SPECTRAL FILTERBANK
# FIXED VERSION
#
# Architecture preserved:
#   EEG
#     ↓
#   6-band learnable sinc filterbank
#     ↓
#   spectral projection
#
# The only change is the cutoff constraint implementation.
# It avoids torch.clamp(Tensor, Tensor, scalar), which is invalid
# in recent PyTorch versions.

class GradientReversalFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = float(lambd)
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None


def grad_reverse(x, lambd: float):
    return GradientReversalFn.apply(x, lambd)


class LearnableSincFilterBank(nn.Module):
    """
    Learnable multi-band Sinc filterbank.

    Frozen project design:
        6 bands initialized around:
            4-8 Hz
            8-12 Hz
            12-16 Hz
            16-22 Hz
            22-30 Hz
            30-38 Hz

    Input:
        x: [B, C, T]

    Output:
        y: [B, C * n_bands, T]

    Notes:
        - Each EEG channel receives the same learnable set of
          band-pass kernels.
        - The six filtered responses are concatenated along
          the channel dimension.
        - The architecture is unchanged.
    """

    def __init__(
        self,
        sfreq=160.0,
        bands=None,
        kernel_size=63,
    ):
        super().__init__()

        if bands is None:
            bands = [
                (4, 8),
                (8, 12),
                (12, 16),
                (16, 22),
                (22, 30),
                (30, 38),
            ]

        self.sfreq = float(sfreq)
        self.n_bands = len(bands)

        kernel_size = int(kernel_size)
        if kernel_size % 2 == 0:
            kernel_size += 1

        self.kernel_size = kernel_size

        # Fixed Hamming window.
        self.register_buffer(
            "window",
            torch.hamming_window(
                self.kernel_size,
                periodic=False,
                dtype=torch.float32,
            ),
        )

        lows, highs = zip(*bands)

        self.low = nn.Parameter(
            torch.tensor(lows, dtype=torch.float32)
        )

        self.high = nn.Parameter(
            torch.tensor(highs, dtype=torch.float32)
        )

    def _kernel(self, low, high, device):
        """
        Construct one differentiable band-pass Sinc kernel.

        IMPORTANT:
        We deliberately avoid:

            torch.clamp(high, low + 1.0, scalar)

        because PyTorch does not support mixed Tensor/scalar
        bounds for that overload.
        """

        n = (self.kernel_size - 1) // 2

        t = (
            torch.arange(
                -n,
                n + 1,
                device=device,
                dtype=torch.float32,
            )
            / self.sfreq
        )

        # ---------------------------------------------------------
        # Valid cutoff limits
        # ---------------------------------------------------------
        nyquist = self.sfreq / 2.0

        min_low = 1.0
        max_low = nyquist - 4.0

        min_high = 2.0
        max_high = nyquist - 1.0

        # Clamp low.
        low = torch.clamp(
            low,
            min=min_low,
            max=max_low,
        )

        # Clamp high using scalar bounds first.
        high = torch.clamp(
            high,
            min=min_high,
            max=max_high,
        )

        # Ensure:
        #
        #       high >= low + 1 Hz
        #
        # torch.maximum supports Tensor-vs-Tensor bounds correctly.
        high = torch.maximum(
            high,
            low + 1.0,
        )

        # Safety in case of unusual parameter excursions.
        high = torch.minimum(
            high,
            torch.tensor(
                max_high,
                device=device,
                dtype=torch.float32,
            ),
        )

        # ---------------------------------------------------------
        # Band-pass Sinc kernel
        #
        # h(t) = 2*f_high*sinc(2*f_high*t)
        #      -2*f_low*sinc(2*f_low*t)
        # ---------------------------------------------------------
        bp = (
            2.0 * high * torch.sinc(2.0 * high * t)
            -
            2.0 * low * torch.sinc(2.0 * low * t)
        )

        # Windowing.
        bp = bp * self.window.to(device)

        # Normalize each filter for numerical stability.
        bp = bp / (bp.abs().sum() + 1e-6)

        return bp

    def forward(self, x):
        """
        Parameters
        ----------
        x : torch.Tensor
            Shape [B, C, T]

        Returns
        -------
        torch.Tensor
            Shape [B, C*n_bands, T]
        """

        B, C, T = x.shape

        # Build all six learnable filters.
        kernels = torch.stack(
            [
                self._kernel(
                    self.low[i],
                    self.high[i],
                    x.device,
                )
                for i in range(self.n_bands)
            ],
            dim=0,
        )
        # kernels:
        # [n_bands, kernel_size]

        # ---------------------------------------------------------
        # Apply all six filters independently to every EEG channel.
        #
        # Input:
        #   [B, C, T]
        #
        # Reshape:
        #   [B*C, 1, T]
        #
        # Kernel:
        #   [6, 1, K]
        #
        # Output:
        #   [B*C, 6, T]
        # ---------------------------------------------------------
        y = F.conv1d(
            x.reshape(B * C, 1, T),
            kernels[:, None, :],
            padding=self.kernel_size // 2,
        )

        # ---------------------------------------------------------
        # Restore EEG-channel structure.
        #
        # [B*C, 6, T]
        #      ↓
        # [B, C*6, T]
        # ---------------------------------------------------------
        y = y.reshape(
            B,
            C * self.n_bands,
            T,
        )

        return y


# -------------------------------------------------------------
# Filterbank unit test
# -------------------------------------------------------------
if "__name__" == "__main__":
    pass


_filter_demo = LearnableSincFilterBank(
    sfreq=TARGET_SFREQ,
    bands=[
        (4, 8),
        (8, 12),
        (12, 16),
        (16, 22),
        (22, 30),
        (30, 38),
    ],
    kernel_size=63,
).to(DEVICE)

with torch.no_grad():
    _x_demo = torch.randn(
        2,
        N_CHANNELS,
        640,
        device=DEVICE,
    )

    _y_demo = _filter_demo(_x_demo)

print("LearnableSincFilterBank test")
print("Input shape :", tuple(_x_demo.shape))
print("Output shape:", tuple(_y_demo.shape))
print(
    "Expected    :",
    (2, N_CHANNELS * 6, 640),
)

del _filter_demo, _x_demo, _y_demo
gc.collect()

LearnableSincFilterBank test
Input shape : (2, 22, 640)
Output shape: (2, 132, 640)
Expected    : (2, 132, 640)


1581

In [29]:
# CELL 13 — TEMPORAL / SPATIAL / ATTENTION BUILDING BLOCKS

class ConvBNAct(nn.Module):
    def __init__(self, cin, cout, kernel, stride=1, groups=1, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(cin, cout, kernel, stride=stride, padding=kernel//2, groups=groups, bias=False),
            nn.BatchNorm1d(cout),
            nn.GELU(),
            nn.Dropout(dropout),
        )
    def forward(self, x): return self.net(x)

class MultiScaleTemporalCNN(nn.Module):
    def __init__(self, cin, out_each=24, dropout=0.1):
        super().__init__()
        self.branches = nn.ModuleList([
            ConvBNAct(cin, out_each, 3, stride=2, groups=1, dropout=dropout),
            ConvBNAct(cin, out_each, 7, stride=2, groups=1, dropout=dropout),
            ConvBNAct(cin, out_each, 15, stride=2, groups=1, dropout=dropout),
            ConvBNAct(cin, out_each, 31, stride=2, groups=1, dropout=dropout),
        ])
    def forward(self, x):
        return torch.cat([b(x) for b in self.branches], dim=1)

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(channels//reduction, 8)
        self.mlp = nn.Sequential(
            nn.Linear(channels, hidden), nn.GELU(), nn.Linear(hidden, channels), nn.Sigmoid()
        )
    def forward(self, x):
        # [B,C,T]
        z = x.mean(dim=-1)
        a = self.mlp(z).unsqueeze(-1)
        return x * a

class ResidualTCNBlock(nn.Module):
    def __init__(self, channels, dilation=1, dropout=0.1):
        super().__init__()
        k=3
        p=dilation*(k-1)//2
        self.conv1 = nn.Conv1d(channels, channels, k, padding=p, dilation=dilation, bias=False)
        self.bn1 = nn.BatchNorm1d(channels)
        self.conv2 = nn.Conv1d(channels, channels, k, padding=p, dilation=dilation, bias=False)
        self.bn2 = nn.BatchNorm1d(channels)
        self.act = nn.GELU()
        self.drop = nn.Dropout(dropout)
    def forward(self,x):
        y=self.act(self.bn1(self.conv1(x)))
        y=self.drop(y)
        y=self.bn2(self.conv2(y))
        return self.act(x+y)

class AttentionPooling(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.score = nn.Linear(dim,1)
    def forward(self,x):
        # [B,L,D]
        a=torch.softmax(self.score(x).squeeze(-1),dim=-1).unsqueeze(-1)
        return (x*a).sum(dim=1)

In [30]:
# CELL 14 — USDA-NET MODEL
# FIXED / ROBUST VERSION
#
# Architecture preserved exactly:
#
# INPUT
#   ↓
# Multi-Scale Temporal CNN
#   +
# Learnable Spectral Filterbank
#   ↓
# Feature Fusion
#   ↓
# Spatial Encoder
#   ↓
# Channel Attention
#   ↓
# Residual TCN
#   ↓
# Transformer × 3
#   ↓
# Attention Pooling
#   ↓
# 128-D embedding
#   ├── Class head
#   ├── DANN domain head
#   └── Prototype representation
#
# No architectural component has been removed.


class USDANet(nn.Module):

    def __init__(
        self,
        n_channels=N_CHANNELS,
        n_classes=3,
        n_domains=2,
        embed_dim=128,
        dropout=0.2,
    ):
        super().__init__()

        self.n_channels = int(n_channels)
        self.n_classes = int(n_classes)
        self.n_domains = int(n_domains)
        self.embed_dim = int(embed_dim)

        # =========================================================
        # 1. DATASET / CHANNEL ENTRY PROJECTION
        # =========================================================
        self.input_proj = nn.Sequential(
            nn.Conv1d(
                self.n_channels,
                32,
                kernel_size=1,
                bias=False,
            ),
            nn.BatchNorm1d(32),
            nn.GELU(),
        )

        # =========================================================
        # 2. MULTI-SCALE TEMPORAL ENCODER
        # =========================================================
        self.temporal = MultiScaleTemporalCNN(
            32,
            out_each=24,
            dropout=dropout,
        )

        temporal_dim = 96

        # =========================================================
        # 3. LEARNABLE SPECTRAL FILTERBANK
        # =========================================================
        self.spectral = LearnableSincFilterBank(
            sfreq=TARGET_SFREQ,
            bands=[
                (4, 8),
                (8, 12),
                (12, 16),
                (16, 22),
                (22, 30),
                (30, 38),
            ],
            kernel_size=63,
        )

        # Six spectral responses per EEG channel.
        self.spectral_proj = nn.Sequential(
            nn.Conv1d(
                self.n_channels * 6,
                32,
                kernel_size=1,
                bias=False,
            ),
            nn.BatchNorm1d(32),
            nn.GELU(),
        )

        # =========================================================
        # 4. TEMPORAL + SPECTRAL FEATURE FUSION
        # =========================================================
        self.fusion = nn.Sequential(
            nn.Conv1d(
                temporal_dim + 32,
                128,
                kernel_size=1,
                bias=False,
            ),
            nn.BatchNorm1d(128),
            nn.GELU(),
        )

        # =========================================================
        # 5. SPATIAL ENCODER
        # =========================================================
        self.spatial_depthwise = nn.Sequential(
            nn.Conv1d(
                128,
                128,
                kernel_size=9,
                padding=4,
                groups=128,
                bias=False,
            ),
            nn.BatchNorm1d(128),
            nn.GELU(),
        )

        # =========================================================
        # 6. CHANNEL ATTENTION
        # =========================================================
        self.channel_attention = ChannelAttention(128)

        # =========================================================
        # 7. RESIDUAL TCN
        # =========================================================
        self.tcn = nn.Sequential(
            ResidualTCNBlock(
                128,
                dilation=1,
                dropout=dropout,
            ),
            ResidualTCNBlock(
                128,
                dilation=2,
                dropout=dropout,
            ),
            ResidualTCNBlock(
                128,
                dilation=4,
                dropout=dropout,
            ),
        )

        # =========================================================
        # 8. TRANSFORMER
        # =========================================================
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=128,
            nhead=4,
            dim_feedforward=256,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=3,
        )

        # =========================================================
        # 9. TOKEN REDUCTION
        # =========================================================
        self.token_pool = nn.AvgPool1d(
            kernel_size=2,
            stride=2,
        )

        # =========================================================
        # 10. POSITIONAL EMBEDDING
        # =========================================================
        #
        # Input = 640 samples.
        # TCN / temporal branch reduces the temporal dimension.
        # This buffer deliberately has enough capacity.
        #
        self.max_transformer_tokens = 160

        self.positional = nn.Parameter(
            torch.zeros(
                1,
                self.max_transformer_tokens,
                128,
            )
        )

        nn.init.trunc_normal_(
            self.positional,
            std=0.02,
        )

        # =========================================================
        # 11. ATTENTION POOLING
        # =========================================================
        self.pool = AttentionPooling(128)

        # =========================================================
        # 12. FINAL EMBEDDING NORMALIZATION
        # =========================================================
        self.embedding_norm = nn.LayerNorm(128)

        # =========================================================
        # 13. CLASSIFICATION HEAD
        # =========================================================
        self.class_head = nn.Sequential(
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, self.n_classes),
        )

        # =========================================================
        # 14. DANN SUBJECT / DOMAIN HEAD
        # =========================================================
        self.domain_head = nn.Sequential(
            nn.Linear(128, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, self.n_domains),
        )

        # =========================================================
        # 15. CLASS PROTOTYPES
        # =========================================================
        self.prototypes = nn.Parameter(
            torch.randn(
                self.n_classes,
                128,
            ) * 0.02
        )

    def forward(
        self,
        x,
        grl_lambda=0.0,
        return_features=True,
    ):
        """
        Input
        -----
        x : [B, C, T]

        Output
        ------
        dict:
            embedding
            logits
            domain_logits
            prototypes
        """

        # =========================================================
        # INPUT
        # =========================================================
        # x = [B, 22, 640]

        # =========================================================
        # TEMPORAL BRANCH
        # =========================================================
        t = self.temporal(
            self.input_proj(x)
        )

        # t shape:
        # [B, 96, T_temporal]

        # =========================================================
        # SPECTRAL BRANCH
        # =========================================================
        spectral_features = self.spectral(x)

        # spectral_features:
        # [B, 22*6, 640]

        s = self.spectral_proj(
            spectral_features
        )

        # s:
        # [B, 32, 640]

        # =========================================================
        # TEMPORAL ALIGNMENT
        # =========================================================
        #
        # Temporal branch is downsampled.
        # Spectral branch retains the full temporal length.
        #
        # We therefore adaptively pool the spectral branch
        # to exactly the temporal length.
        #
        if s.shape[-1] != t.shape[-1]:
            s = F.adaptive_avg_pool1d(
                s,
                t.shape[-1],
            )

        # =========================================================
        # FEATURE FUSION
        # =========================================================
        z = torch.cat(
            [t, s],
            dim=1,
        )

        # [B, 128, L]

        z = self.fusion(z)

        # =========================================================
        # SPATIAL ENCODER
        # =========================================================
        z = self.spatial_depthwise(z)

        # =========================================================
        # CHANNEL ATTENTION
        # =========================================================
        z = self.channel_attention(z)

        # =========================================================
        # RESIDUAL TCN
        # =========================================================
        z = self.tcn(z)

        # =========================================================
        # TOKEN REDUCTION
        # =========================================================
        #
        # [B,128,L]
        #     ↓
        # [B,128,L/2]
        #
        z = self.token_pool(z)

        # =========================================================
        # TRANSFORMER INPUT
        # =========================================================
        #
        # [B,128,L]
        #     ↓ transpose
        # [B,L,128]
        #
        z = z.transpose(1, 2)

        L = z.shape[1]

        # =========================================================
        # POSITIONAL EMBEDDING
        # =========================================================
        if L <= self.max_transformer_tokens:

            z = z + self.positional[:, :L, :]

        else:
            # Very defensive path for future different window sizes.
            # Interpolate positional embeddings rather than crashing.
            pos = F.interpolate(
                self.positional.transpose(1, 2),
                size=L,
                mode="linear",
                align_corners=False,
            ).transpose(1, 2)

            z = z + pos

        # =========================================================
        # TRANSFORMER × 3
        # =========================================================
        z = self.transformer(z)

        # =========================================================
        # ATTENTION POOLING
        # =========================================================
        emb = self.pool(z)

        # =========================================================
        # EMBEDDING NORMALIZATION
        # =========================================================
        emb = self.embedding_norm(emb)

        # =========================================================
        # CLASS PREDICTION
        # =========================================================
        logits = self.class_head(emb)

        # =========================================================
        # DOMAIN ADVERSARIAL BRANCH
        # =========================================================
        #
        # Gradient reversal happens here.
        #
        dfeat = grad_reverse(
            emb,
            grl_lambda,
        )

        domain_logits = self.domain_head(
            dfeat
        )

        # =========================================================
        # NORMALIZED CLASS PROTOTYPES
        # =========================================================
        normalized_prototypes = F.normalize(
            self.prototypes,
            dim=-1,
        )

        return {
            "embedding": emb,
            "logits": logits,
            "domain_logits": domain_logits,
            "prototypes": normalized_prototypes,
        }


# =============================================================
# FORWARD-SHAPE SMOKE TEST
# =============================================================

_demo_model = USDANet(
    n_channels=N_CHANNELS,
    n_classes=3,
    n_domains=5,
).to(DEVICE)

_demo_input = torch.randn(
    4,
    N_CHANNELS,
    640,
    device=DEVICE,
)

with torch.no_grad():

    _demo_out = _demo_model(
        _demo_input,
        grl_lambda=0.5,
    )

print("=" * 70)
print("USDA-NET FORWARD TEST")
print("=" * 70)

for key, value in _demo_out.items():

    if hasattr(value, "shape"):
        print(
            f"{key:20s}: "
            f"{tuple(value.shape)}"
        )
    else:
        print(
            f"{key:20s}: "
            f"{type(value)}"
        )

print("=" * 70)

assert _demo_out["embedding"].shape == (4, 128)
assert _demo_out["logits"].shape == (4, 3)
assert _demo_out["domain_logits"].shape == (4, 5)
assert _demo_out["prototypes"].shape == (3, 128)

print("✓ USDA-Net forward test PASSED")

del _demo_model
del _demo_input
del _demo_out

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

USDA-NET FORWARD TEST
embedding           : (4, 128)
logits              : (4, 3)
domain_logits       : (4, 5)
prototypes          : (3, 128)
✓ USDA-Net forward test PASSED


/var/folders/kq/cbr0cvmd3sq82p6gbfd_31980000gn/T/ipykernel_2482/566835166.py:171: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


In [31]:
# CELL 15 — DATASET CLASS + TRAINING AUGMENTATION

class EEGArrayDataset(Dataset):
    def __init__(self, path, indices, normalizer=None, augment=False):
        self.path = Path(path)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.normalizer = normalizer
        self.augment = augment
        self._h5 = None
    def _open(self):
        if self._h5 is None:
            self._h5 = h5py.File(self.path, "r")
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        self._open()
        idx = int(self.indices[i])
        # Need cache-global index, not the local EEGMMIDB subset position.
        X = np.asarray(self._h5["X"][idx], dtype=np.float32)
        meta_idx = idx
        label_name = self._h5["metadata"]["harmonized_class"][meta_idx]
        if isinstance(label_name, bytes): label_name = label_name.decode("utf-8")
        y = CLASS_TO_ID[str(label_name)]
        if self.normalizer is not None:
            X = self.normalizer.transform(X[None])[0]
        if self.augment:
            X = augment_eeg(X)
        return torch.from_numpy(X), torch.tensor(y, dtype=torch.long), idx
    def __del__(self):
        try:
            if self._h5 is not None: self._h5.close()
        except Exception:
            pass


def augment_eeg(x: np.ndarray) -> np.ndarray:
    x = x.copy()
    # Mild, label-preserving training-only augmentation.
    if np.random.rand() < 0.5:
        x += np.random.normal(0.0, 0.01*np.std(x), size=x.shape).astype(np.float32)
    if np.random.rand() < 0.35:
        scale = np.random.uniform(0.9, 1.1)
        x *= scale
    if np.random.rand() < 0.30:
        c = np.random.randint(0, x.shape[0])
        x[c] = 0.0
    if np.random.rand() < 0.25:
        t0 = np.random.randint(0, max(1, x.shape[1]-16))
        width = np.random.randint(8, 32)
        x[:, t0:min(x.shape[1],t0+width)] *= 0.0
    return x.astype(np.float32)

print("Dataset and augmentation interface ready.")

Dataset and augmentation interface ready.


In [32]:
# CELL 16 — EFFICIENT INDEX MAPPING + BALANCED BATCH SAMPLER HELPERS

# cache_indices maps the EEGMMIDB-only row order to indices in the HDF5 cache.
meta_by_cache_index = meta_df.copy()
meta_by_cache_index["cache_index"] = cache_indices

subject_to_id = {s:i for i,s in enumerate(sorted(meta_df.subject.unique()))}
meta_by_cache_index["subject_id"] = meta_by_cache_index.subject.map(subject_to_id).astype(int)
meta_by_cache_index["class_id"] = meta_by_cache_index.harmonized_class.map(CLASS_TO_ID).astype(int)

print("EEGMMIDB subjects:", len(subject_to_id))
print("First subject IDs:", list(subject_to_id.items())[:10])


def read_X(cache_indices_requested: np.ndarray) -> np.ndarray:
    inds = np.asarray(cache_indices_requested, dtype=np.int64)
    if len(inds) == 0:
        return np.empty((0,22,640), dtype=np.float32)
    with h5py.File(ACTIVE_CACHE, "r") as h5:
        order = np.argsort(inds)
        sorted_inds = inds[order]
        xs = np.asarray(h5["X"][sorted_inds], dtype=np.float32)
        inv = np.argsort(order)
        return xs[inv]


def read_meta_rows(requested_indices):
    requested = np.asarray(requested_indices, dtype=np.int64)
    return meta_by_cache_index.set_index("cache_index").loc[requested].reset_index()

print("Data access helpers ready.")

EEGMMIDB subjects: 109
First subject IDs: [('S001', 0), ('S002', 1), ('S003', 2), ('S004', 3), ('S005', 4), ('S006', 5), ('S007', 6), ('S008', 7), ('S009', 8), ('S010', 9)]
Data access helpers ready.


In [33]:
# CELL 17 — LOSS FUNCTIONS

class SupervisedContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.1):
        super().__init__()
        self.temperature = temperature
    def forward(self, features, labels):
        # features [B,D]
        z = F.normalize(features, dim=-1)
        logits = z @ z.T / self.temperature
        logits = logits - logits.max(dim=1, keepdim=True).values.detach()
        labels = labels.view(-1,1)
        mask = torch.eq(labels, labels.T).float().to(z.device)
        self_mask = torch.eye(len(labels), device=z.device)
        mask = mask - self_mask
        exp_logits = torch.exp(logits) * (1-self_mask)
        log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-8)
        pos_count = mask.sum(dim=1)
        valid = pos_count > 0
        if valid.sum() == 0:
            return logits.new_tensor(0.0)
        mean_log_prob_pos = (mask * log_prob).sum(dim=1) / (pos_count + 1e-8)
        return (-mean_log_prob_pos[valid]).mean()


def prototype_loss(embeddings, labels, prototypes, margin=0.0):
    z = F.normalize(embeddings, dim=-1)
    p = F.normalize(prototypes, dim=-1)
    sim = z @ p.T
    target_sim = sim.gather(1, labels[:,None]).squeeze(1)
    if margin > 0:
        logits = sim / 0.1
        return F.cross_entropy(logits, labels)
    return (1.0 - target_sim).mean()

contrastive_criterion = SupervisedContrastiveLoss(temperature=0.10)
print("Loss functions ready.")

Loss functions ready.


In [34]:
# CELL 18 — FOLD-SAFE METRICS / LEAKAGE GUARDS

def compute_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "kappa": float(cohen_kappa_score(y_true, y_pred)),
        "recall_left": float(f1_score(y_true, y_pred, labels=[0], average=None, zero_division=0)[0]),
        "recall_right": float(f1_score(y_true, y_pred, labels=[1], average=None, zero_division=0)[0]),
        "recall_feet": float(f1_score(y_true, y_pred, labels=[2], average=None, zero_division=0)[0]),
    }


def build_loso_fold(target_subject: str):
    all_subjects = sorted(meta_df.subject.unique())
    source_subjects = [s for s in all_subjects if s != target_subject]
    # Deterministic source-only validation subject. Target is never eligible.
    val_subject = source_subjects[(SEED + int(re.search(r"(\d+)$", source_subjects[0]).group(1) if source_subjects else 0)) % len(source_subjects)]
    # More useful deterministic selection: hash all source IDs.
    val_subject = sorted(source_subjects, key=lambda s: hashlib.sha1(f"{SEED}:{s}".encode()).hexdigest())[0]
    train_subjects = [s for s in source_subjects if s != val_subject]
    assert target_subject not in train_subjects
    assert target_subject != val_subject
    return train_subjects, [val_subject], [target_subject]


def assert_fold_subject_disjoint(train_subjects, val_subjects, test_subjects):
    a,b,c = set(train_subjects),set(val_subjects),set(test_subjects)
    assert not (a & b)
    assert not (a & c)
    assert not (b & c)
    return True

print("LOSO helpers ready.")

LOSO helpers ready.


In [35]:
# CELL 19 — DATA INDEX EXTRACTION + SOURCE-ONLY NORMALIZATION BUILDER

def indices_for_subjects(subjects):
    subjects = set(subjects)
    return meta_by_cache_index.loc[meta_by_cache_index.subject.isin(subjects), "cache_index"].to_numpy(dtype=np.int64)


def fit_fold_normalizer(train_indices):
    X_train = read_X(train_indices)
    subjects = meta_by_cache_index.set_index("cache_index").loc[train_indices, "subject"].tolist()
    normalizer = SourceOnlyRobustNormalizer().fit(X_train, subjects)
    del X_train
    gc.collect()
    return normalizer

print("Index and source-normalization helpers ready.")

Index and source-normalization helpers ready.


In [54]:
# ============================================================
# CELL 20 — USDA-NET TRAINING FUNCTION FOR DANN ABLATION
# ============================================================
#
# IMPORTANT:
# The NETWORK ARCHITECTURE is unchanged.
#
# The ONLY experimental variable is:
#
#     dann_max_lambda
#
# Values for the planned ablation:
#
#     0.10
#     0.20
#     0.30
#
# ============================================================


def train_one_fold(
    target_subject,

    max_epochs=35,

    batch_size=128,

    lr=3e-4,

    weight_decay=1e-4,

    patience=10,

    dann_max_lambda=0.30,

    experiment_name="USDA_Net",
):
    """
    Train one strict LOSO fold.

    The target subject is completely excluded from:

        training
        normalization
        validation
        early stopping
        DANN
        prototype learning
        supervised contrastive learning

    Parameters
    ----------
    target_subject : str
        Held-out subject.

    dann_max_lambda : float
        Maximum DANN gradient-reversal coefficient.

        Ablation values:
            0.10
            0.20
            0.30

    experiment_name : str
        Name used only for result organization.
    """

    # ========================================================
    # 1. REPRODUCIBILITY
    # ========================================================

    seed_everything(
        GLOBAL_SEED
        if "GLOBAL_SEED" in globals()
        else SEED
    )


    # ========================================================
    # 2. SAFETY CHECK
    # ========================================================

    assert (
        0.0 <= dann_max_lambda <= 1.0
    ), (
        "dann_max_lambda must be "
        "between 0 and 1."
    )


    # ========================================================
    # 3. BUILD STRICT LOSO FOLD
    # ========================================================

    (
        train_subjects,
        val_subjects,
        test_subjects,
    ) = build_loso_fold(
        target_subject
    )

    assert_fold_subject_disjoint(
        train_subjects,
        val_subjects,
        test_subjects,
    )

    assert (
        target_subject not in train_subjects
    )

    assert (
        target_subject not in val_subjects
    )

    assert (
        test_subjects == [target_subject]
    )


    # ========================================================
    # 4. INDICES
    # ========================================================

    train_idx = indices_for_subjects(
        train_subjects
    )

    val_idx = indices_for_subjects(
        val_subjects
    )

    test_idx = indices_for_subjects(
        test_subjects
    )


    assert len(train_idx) > 0
    assert len(val_idx) > 0
    assert len(test_idx) > 0


    # ========================================================
    # 5. SOURCE-ONLY NORMALIZATION
    # ========================================================

    normalizer = fit_fold_normalizer(
        train_idx
    )

    normalizer.assert_target_excluded(
        target_subject
    )

    for subject in val_subjects:

        assert (
            str(subject)
            not in
            set(
                normalizer.fitted_subjects_
            )
        )


    # ========================================================
    # 6. DOMAIN LABEL SPACE
    # ========================================================

    n_domains = len(
        train_subjects
    )

    train_subject_id = {
        subject: idx
        for idx, subject
        in enumerate(
            train_subjects
        )
    }

    cache_to_subject = (
        meta_by_cache_index
        .set_index(
            "cache_index"
        )["subject"]
        .to_dict()
    )


    # ========================================================
    # 7. CREATE USDA-NET
    # ========================================================

    model = USDANet(
        n_channels=N_CHANNELS,
        n_classes=len(
            PRIMARY_CLASSES
        ),
        n_domains=n_domains,
        embed_dim=128,
        dropout=0.20,
    ).to(DEVICE)


    # ========================================================
    # 8. OPTIMIZER
    # ========================================================

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )


    # ========================================================
    # 9. COSINE LR
    # ========================================================

    scheduler = (
        torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=max_epochs,
            eta_min=lr * 0.10,
        )
    )


    # ========================================================
    # 10. LOSSES
    # ========================================================

    classification_criterion = (
        nn.CrossEntropyLoss(
            label_smoothing=0.05
        )
    )

    contrastive_criterion = (
        SupervisedContrastiveLoss(
            temperature=0.10
        )
    )


    # ========================================================
    # 11. DATA LOADERS
    # ========================================================

    (
        train_loader,
        val_loader,
    ) = make_loaders(
        train_idx=train_idx,
        val_idx=val_idx,
        normalizer=normalizer,
        train_subjects=train_subjects,
        batch_size=batch_size,
    )


    # ========================================================
    # 12. BEST MODEL
    # ========================================================

    best_val_score = -np.inf

    best_state = None

    best_epoch = -1

    wait = 0

    history = []


    # ========================================================
    # 13. TRAINING
    # ========================================================

    for epoch in range(
        1,
        max_epochs + 1,
    ):

        model.train()


        # ====================================================
        # DANN SCHEDULE
        # ====================================================

        lambda_d = dann_lambda(
            epoch=epoch,
            max_epochs=max_epochs,
            warmup_fraction=0.25,
            gamma=5.0,
            max_lambda=dann_max_lambda,
        )

        assert (
            0.0 <= lambda_d
            <= dann_max_lambda
        )


        # ====================================================
        # MINI-BATCH LOOP
        # ====================================================

        totals = Counter()

        steps = 0


        for (
            xb,
            yb,
            cache_ids
        ) in train_loader:

            xb = xb.to(DEVICE)

            yb = yb.to(DEVICE)


            # ------------------------------------------------
            # Source-domain labels
            # ------------------------------------------------

            subject_labels = torch.tensor(
                [
                    train_subject_id[
                        cache_to_subject[
                            int(cache_id)
                        ]
                    ]
                    for cache_id
                    in cache_ids.tolist()
                ],
                dtype=torch.long,
                device=DEVICE,
            )


            # ------------------------------------------------
            # USDA-Net forward
            # ------------------------------------------------

            out = model(
                xb,
                grl_lambda=lambda_d,
            )


            # ------------------------------------------------
            # Classification
            # ------------------------------------------------

            loss_task = (
                classification_criterion(
                    out["logits"],
                    yb,
                )
            )


            # ------------------------------------------------
            # DANN
            # ------------------------------------------------

            loss_domain = (
                classification_criterion(
                    out["domain_logits"],
                    subject_labels,
                )
            )


            # ------------------------------------------------
            # Prototype
            # ------------------------------------------------

            loss_proto = (
                prototype_loss(
                    out["embedding"],
                    yb,
                    model.prototypes,
                    margin=1.0,
                )
            )


            # ------------------------------------------------
            # Supervised contrastive
            # ------------------------------------------------

            loss_supcon = (
                contrastive_criterion(
                    out["embedding"],
                    yb,
                )
            )


            # ------------------------------------------------
            # TOTAL LOSS
            # ------------------------------------------------
            #
            # Same weights for every ablation.
            #
            # Only lambda_d changes the GRL behavior.
            #
            # ------------------------------------------------

            loss = (
                1.00 * loss_task
                +
                0.30 * loss_domain
                +
                0.05 * loss_proto
                +
                0.10 * loss_supcon
            )


            # ------------------------------------------------
            # Backpropagation
            # ------------------------------------------------

            optimizer.zero_grad(
                set_to_none=True
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0,
            )

            optimizer.step()


            # ------------------------------------------------
            # Accumulate
            # ------------------------------------------------

            totals["loss"] += float(
                loss.detach().item()
            )

            totals["task"] += float(
                loss_task.detach().item()
            )

            totals["domain"] += float(
                loss_domain.detach().item()
            )

            totals["proto"] += float(
                loss_proto.detach().item()
            )

            totals["supcon"] += float(
                loss_supcon.detach().item()
            )

            steps += 1


        # ====================================================
        # LR SCHEDULER
        # ====================================================

        scheduler.step()


        train_avg = {
            key:
                value / max(
                    steps,
                    1,
                )
            for key, value
            in totals.items()
        }


        # ====================================================
        # VALIDATION
        # ====================================================

        (
            val_metrics,
            val_true,
            val_pred,
            val_loss,
        ) = evaluate_model(
            model,
            val_loader,
        )


        current_lr = float(
            optimizer
            .param_groups[0]["lr"]
        )


        # ====================================================
        # RECORD HISTORY
        # ====================================================

        row = {

            "experiment":
                experiment_name,

            "target_subject":
                target_subject,

            "dann_max_lambda":
                dann_max_lambda,

            "epoch":
                epoch,

            "lambda_domain":
                lambda_d,

            "learning_rate":
                current_lr,

            "train_total":
                train_avg["loss"],

            "train_task":
                train_avg["task"],

            "train_domain":
                train_avg["domain"],

            "train_prototype":
                train_avg["proto"],

            "train_supcon":
                train_avg["supcon"],

            "val_loss":
                val_loss,

            "val_accuracy":
                val_metrics[
                    "accuracy"
                ],

            "val_balanced_accuracy":
                val_metrics[
                    "balanced_accuracy"
                ],

            "val_macro_f1":
                val_metrics[
                    "macro_f1"
                ],

            "val_kappa":
                val_metrics[
                    "kappa"
                ],
        }

        history.append(row)


        # ====================================================
        # MODEL SELECTION
        # ====================================================

        val_score = (
            val_metrics[
                "balanced_accuracy"
            ]
        )


        if val_score > (
            best_val_score
            + 1e-5
        ):

            best_val_score = (
                val_score
            )

            best_epoch = (
                epoch
            )

            best_state = {
                key:
                    value
                    .detach()
                    .cpu()
                    .clone()

                for key, value
                in model.state_dict()
                .items()
            }

            wait = 0

        else:

            wait += 1


        # ====================================================
        # LOG
        # ====================================================

        if (
            epoch == 1
            or epoch % 5 == 0
            or epoch == max_epochs
        ):

            print(
                f"[{experiment_name}] "
                f"[{target_subject}] "
                f"ep={epoch:03d} "
                f"λmax={dann_max_lambda:.2f} "
                f"λ={lambda_d:.3f} "
                f"lr={current_lr:.2e} "
                f"task={train_avg['task']:.4f} "
                f"dom={train_avg['domain']:.4f} "
                f"proto={train_avg['proto']:.4f} "
                f"supcon={train_avg['supcon']:.4f} "
                f"valBAcc="
                f"{val_metrics['balanced_accuracy']*100:.2f}% "
                f"valF1="
                f"{val_metrics['macro_f1']*100:.2f}%"
            )


        # ====================================================
        # EARLY STOPPING
        # ====================================================

        if wait >= patience:

            print(
                f"[{experiment_name}] "
                f"Early stopping at epoch "
                f"{epoch}; "
                f"best epoch="
                f"{best_epoch}; "
                f"best val BAcc="
                f"{best_val_score*100:.2f}%"
            )

            break


    # ========================================================
    # RESTORE BEST MODEL
    # ========================================================

    if best_state is None:

        raise RuntimeError(
            f"No best state for "
            f"{experiment_name}"
        )

    model.load_state_dict(
        best_state
    )


    # ========================================================
    # TARGET TEST
    # ========================================================
    #
    # Target labels are accessed ONLY here.
    # ========================================================

    test_ds = EEGArrayDataset(
        ACTIVE_CACHE,
        test_idx,
        normalizer=normalizer,
        augment=False,
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )


    (
        test_metrics,
        y_true,
        y_pred,
        test_loss,
    ) = evaluate_model(
        model,
        test_loader,
    )


    # ========================================================
    # FINAL LEAKAGE ASSERTIONS
    # ========================================================

    assert (
        target_subject
        not in
        normalizer.fitted_subjects_
    )

    assert (
        target_subject
        not in train_subjects
    )

    assert (
        target_subject
        not in val_subjects
    )


    # ========================================================
    # RETURN
    # ========================================================

    return {

        "experiment":
            experiment_name,

        "dann_max_lambda":
            dann_max_lambda,

        "target_subject":
            target_subject,

        "train_subjects":
            train_subjects,

        "val_subjects":
            val_subjects,

        "test_subjects":
            test_subjects,

        "best_epoch":
            best_epoch,

        "best_val_balanced_accuracy":
            best_val_score,

        "test_metrics":
            test_metrics,

        "test_loss":
            test_loss,

        "history":
            pd.DataFrame(
                history
            ),

        "y_true":
            y_true,

        "y_pred":
            y_pred,

        "normalizer":
            normalizer,

        "model":
            model,

        "n_domains":
            n_domains,
    }


print("=" * 78)
print("DANN ABLATION TRAINING FUNCTION READY")
print("=" * 78)
print("Architecture : USDA-Net")
print("Ablated item : DANN maximum lambda")
print("Values       : 0.10 / 0.20 / 0.30")

DANN ABLATION TRAINING FUNCTION READY
Architecture : USDA-Net
Ablated item : DANN maximum lambda
Values       : 0.10 / 0.20 / 0.30


In [55]:
# ============================================================
# CELL 21 — DANN ABLATION SCHEDULE CHECK
# ============================================================

ABLATION_LAMBDAS = [
    0.10,
    0.20,
    0.30,
]

ABLATION_EPOCHS = 35


print("=" * 78)
print("DANN ABLATION SCHEDULES")
print("=" * 78)


for max_lambda in ABLATION_LAMBDAS:

    rows = []

    for epoch in range(
        1,
        ABLATION_EPOCHS + 1,
    ):

        lam = dann_lambda(
            epoch=epoch,
            max_epochs=ABLATION_EPOCHS,
            warmup_fraction=0.25,
            gamma=5.0,
            max_lambda=max_lambda,
        )

        rows.append(
            {
                "epoch":
                    epoch,

                "lambda":
                    lam,
            }
        )


    tmp = pd.DataFrame(
        rows
    )

    print(
        f"\nDANN max λ = {max_lambda:.2f}"
    )

    display(
        tmp[
            tmp["epoch"].isin(
                [
                    1,
                    5,
                    10,
                    15,
                    20,
                    25,
                    30,
                    35,
                ]
            )
        ]
    )

    assert (
        tmp["lambda"].min()
        >= 0.0
    )

    assert (
        tmp["lambda"].max()
        <= max_lambda + 1e-8
    )


print(
    "\n✓ All three DANN schedules are valid."
)

DANN ABLATION SCHEDULES

DANN max λ = 0.10


,epoch,lambda
0,1,0.000000
4,5,0.000000
9,10,0.009586
14,15,0.052043
19,20,0.078478
24,25,0.091186
29,30,0.096536
34,35,0.098661



DANN max λ = 0.20


,epoch,lambda
0,1,0.000000
4,5,0.000000
9,10,0.019172
14,15,0.104085
19,20,0.156956
24,25,0.182372
29,30,0.193072
34,35,0.197323



DANN max λ = 0.30


,epoch,lambda
0,1,0.000000
4,5,0.000000
9,10,0.028758
14,15,0.156128
19,20,0.235434
24,25,0.273558
29,30,0.289608
34,35,0.295984



✓ All three DANN schedules are valid.


## Full strict LOSO experiment

For EEGMMIDB, the primary experiment holds out **one subject at a time**. The held-out subject is absent from:

- model fitting;
- source normalization;
- validation;
- augmentation;
- early stopping/model selection;
- domain-classifier fitting;
- prototype fitting.

The notebook can resume fold-by-fold because every fold is saved immediately.

In [56]:
# ============================================================
# CELL 22 — 3-WAY DANN ABLATION CONFIGURATION
# ============================================================

ABLATION_TARGET = "S001"

ABLATION_EPOCHS = 35

ABLATION_BATCH_SIZE = 128

ABLATION_LR = 3e-4

ABLATION_WEIGHT_DECAY = 1e-4

ABLATION_PATIENCE = 10

ABLATION_LAMBDAS = [
    0.10,
    0.20,
    0.30,
]


ABLATION_ROOT = (
    RESULTS_ROOT /
    "DANN_ablation_S001"
)

ABLATION_HISTORY_ROOT = (
    ABLATION_ROOT /
    "histories"
)

ABLATION_CHECKPOINT_ROOT = (
    ABLATION_ROOT /
    "checkpoints"
)

ABLATION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

ABLATION_HISTORY_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

ABLATION_CHECKPOINT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


print("=" * 78)
print("3-WAY DANN ABLATION")
print("=" * 78)

print(
    "Target subject:",
    ABLATION_TARGET
)

print(
    "DANN λ values:",
    ABLATION_LAMBDAS
)

print(
    "Epochs:",
    ABLATION_EPOCHS
)

print(
    "Batch size:",
    ABLATION_BATCH_SIZE
)

print(
    "Learning rate:",
    ABLATION_LR
)

print(
    "Patience:",
    ABLATION_PATIENCE
)

print(
    "Results root:",
    ABLATION_ROOT
)

assert (
    ABLATION_TARGET
    in set(
        meta_df.subject.unique()
    )
)

print(
    "\n✓ S001 exists in the dataset."
)

3-WAY DANN ABLATION
Target subject: S001
DANN λ values: [0.1, 0.2, 0.3]
Epochs: 35
Batch size: 128
Learning rate: 0.0003
Patience: 10
Results root: /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/results/USDA_Net_PhysioNet_3Class/DANN_ablation_S001

✓ S001 exists in the dataset.


In [58]:
# ============================================================
# CELL 23 — RUN 3-WAY DANN ABLATION ON S001
# ============================================================
#
# EXPERIMENTS:
#
#   A: λmax = 0.10
#   B: λmax = 0.20
#   C: λmax = 0.30
#
# EVERYTHING ELSE IS IDENTICAL.
#
# ============================================================


ablation_results = []

ablation_artifacts = {}


for max_lambda in ABLATION_LAMBDAS:

    experiment_name = (
        f"DANN_lambda_{max_lambda:.2f}"
    )

    print(
        "\n"
        +
        "=" * 78
    )

    print(
        f"STARTING {experiment_name}"
    )

    print(
        f"Target subject = "
        f"{ABLATION_TARGET}"
    )

    print(
        f"Maximum DANN λ = "
        f"{max_lambda:.2f}"
    )

    print(
        "=" * 78
    )


    # --------------------------------------------------------
    # Re-seed before every experiment.
    #
    # This ensures the only intended experimental variable
    # is the DANN maximum lambda.
    # --------------------------------------------------------

    seed_everything(
        GLOBAL_SEED
    )


    start_time = time.time()


    artifact = train_one_fold(

        target_subject=
            ABLATION_TARGET,

        max_epochs=
            ABLATION_EPOCHS,

        batch_size=
            ABLATION_BATCH_SIZE,

        lr=
            ABLATION_LR,

        weight_decay=
            ABLATION_WEIGHT_DECAY,

        patience=
            ABLATION_PATIENCE,

        dann_max_lambda=
            max_lambda,

        experiment_name=
            experiment_name,
    )


    runtime = (
        time.time()
        -
        start_time
    )


    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    metrics = artifact[
        "test_metrics"
    ]


    result_row = {

        "experiment":
            experiment_name,

        "target_subject":
            ABLATION_TARGET,

        "dann_max_lambda":
            max_lambda,

        "best_epoch":
            artifact[
                "best_epoch"
            ],

        "best_val_bacc":
            artifact[
                "best_val_balanced_accuracy"
            ],

        "test_accuracy":
            metrics[
                "accuracy"
            ],

        "test_balanced_accuracy":
            metrics[
                "balanced_accuracy"
            ],

        "test_macro_f1":
            metrics[
                "macro_f1"
            ],

        "test_kappa":
            metrics[
                "kappa"
            ],

        "left_recall":
            metrics[
                "recall_left"
            ],

        "right_recall":
            metrics[
                "recall_right"
            ],

        "feet_recall":
            metrics[
                "recall_feet"
            ],

        "runtime_sec":
            runtime,

    }


    ablation_results.append(
        result_row
    )


    # --------------------------------------------------------
    # Save history
    # --------------------------------------------------------

    history_path = (
        ABLATION_HISTORY_ROOT
        /
        f"{experiment_name}_history.csv"
    )

    artifact[
        "history"
    ].to_csv(
        history_path,
        index=False,
    )


    # --------------------------------------------------------
    # Save checkpoint
    # --------------------------------------------------------

    checkpoint_path = (
        ABLATION_CHECKPOINT_ROOT
        /
        f"{experiment_name}.pt"
    )


    torch.save(
        {
            "experiment":
                experiment_name,

            "target_subject":
                ABLATION_TARGET,

            "dann_max_lambda":
                max_lambda,

            "model_state_dict":
                {
                    key:
                        value.cpu()

                    for key,
                        value
                    in artifact[
                        "model"
                    ]
                    .state_dict()
                    .items()
                },

            "normalizer_median":
                artifact[
                    "normalizer"
                ].median_,

            "normalizer_iqr":
                artifact[
                    "normalizer"
                ].iqr_,

            "config": {
                "epochs":
                    ABLATION_EPOCHS,

                "batch_size":
                    ABLATION_BATCH_SIZE,

                "learning_rate":
                    ABLATION_LR,

                "weight_decay":
                    ABLATION_WEIGHT_DECAY,

                "patience":
                    ABLATION_PATIENCE,

                "architecture":
                    "USDA-Net",

                "dataset":
                    DATASET,

                "classes":
                    PRIMARY_CLASSES,

                "channels":
                    FROZEN_COMMON_CHANNELS,

                "sampling_rate":
                    TARGET_SFREQ,

                "bandpass":
                    [
                        LOW_HZ,
                        HIGH_HZ,
                    ],

                "epoch":
                    [
                        TMIN,
                        TMAX,
                    ],

                "strict_loso":
                    True,
            },
        },

        checkpoint_path,
    )


    ablation_artifacts[
        max_lambda
    ] = artifact


    # --------------------------------------------------------
    # Immediate summary
    # --------------------------------------------------------

    print(
        "\n"
        +
        "-" * 78
    )

    print(
        experiment_name,
        "COMPLETE"
    )

    print(
        "Best validation BAcc:",
        f"{artifact['best_val_balanced_accuracy']*100:.2f}%"
    )

    print(
        "Target accuracy:",
        f"{metrics['accuracy']*100:.2f}%"
    )

    print(
        "Target balanced accuracy:",
        f"{metrics['balanced_accuracy']*100:.2f}%"
    )

    print(
        "Target Macro-F1:",
        f"{metrics['macro_f1']*100:.2f}%"
    )

    print(
        "Runtime:",
        f"{runtime:.1f} sec"
    )

    print(
        "-" * 78
    )


    # --------------------------------------------------------
    # Cleanup
    # --------------------------------------------------------

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# SAVE RESULTS
# ============================================================

ablation_results_df = (
    pd.DataFrame(
        ablation_results
    )
    .sort_values(
        "dann_max_lambda"
    )
    .reset_index(
        drop=True
    )
)


ABLATION_RESULTS_CSV = (
    ABLATION_ROOT /
    "dann_ablation_results.csv"
)


ablation_results_df.to_csv(
    ABLATION_RESULTS_CSV,
    index=False,
)


print(
    "\n"
    +
    "=" * 78
)

print(
    "3-WAY DANN ABLATION COMPLETE"
)

print(
    "=" * 78
)

display(
    ablation_results_df
)


STARTING DANN_lambda_0.10
Target subject = S001
Maximum DANN λ = 0.10


NameError: name 'GLOBAL_SEED' is not defined

In [59]:
# ============================================================
# CELL 23 — RUN 3-WAY DANN ABLATION ON S001
# FIXED: uses existing SEED variable
# ============================================================

# ------------------------------------------------------------
# Reproducibility seed already defined by the project notebook:
#
#     SEED = 20260822
#
# We intentionally do NOT require GLOBAL_SEED.
# ------------------------------------------------------------

ABLATION_SEED = (
    SEED
    if "SEED" in globals()
    else 20260822
)

print(
    "Ablation seed:",
    ABLATION_SEED
)


# ============================================================
# RESULT CONTAINERS
# ============================================================

ablation_results = []

ablation_artifacts = {}


# ============================================================
# RUN THREE DANN SETTINGS
# ============================================================

for max_lambda in ABLATION_LAMBDAS:

    experiment_name = (
        f"DANN_lambda_{max_lambda:.2f}"
    )

    print(
        "\n"
        +
        "=" * 78
    )

    print(
        f"STARTING {experiment_name}"
    )

    print(
        f"Target subject = "
        f"{ABLATION_TARGET}"
    )

    print(
        f"Maximum DANN λ = "
        f"{max_lambda:.2f}"
    )

    print(
        "=" * 78
    )


    # --------------------------------------------------------
    # Re-seed every experiment.
    #
    # This keeps initialization and data-side randomness
    # comparable across the three DANN settings.
    # --------------------------------------------------------

    seed_everything(
        ABLATION_SEED
    )


    # --------------------------------------------------------
    # Runtime
    # --------------------------------------------------------

    start_time = time.time()


    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    artifact = train_one_fold(

        target_subject=
            ABLATION_TARGET,

        max_epochs=
            ABLATION_EPOCHS,

        batch_size=
            ABLATION_BATCH_SIZE,

        lr=
            ABLATION_LR,

        weight_decay=
            ABLATION_WEIGHT_DECAY,

        patience=
            ABLATION_PATIENCE,

        dann_max_lambda=
            max_lambda,

        experiment_name=
            experiment_name,
    )


    runtime = (
        time.time()
        -
        start_time
    )


    # ========================================================
    # TEST METRICS
    # ========================================================

    metrics = artifact[
        "test_metrics"
    ]


    # ========================================================
    # RESULT ROW
    # ========================================================

    result_row = {

        "experiment":
            experiment_name,

        "target_subject":
            ABLATION_TARGET,

        "dann_max_lambda":
            max_lambda,

        "best_epoch":
            artifact[
                "best_epoch"
            ],

        "best_val_bacc":
            artifact[
                "best_val_balanced_accuracy"
            ],

        "test_accuracy":
            metrics[
                "accuracy"
            ],

        "test_balanced_accuracy":
            metrics[
                "balanced_accuracy"
            ],

        "test_macro_f1":
            metrics[
                "macro_f1"
            ],

        "test_kappa":
            metrics[
                "kappa"
            ],

        "left_recall":
            metrics[
                "recall_left"
            ],

        "right_recall":
            metrics[
                "recall_right"
            ],

        "feet_recall":
            metrics[
                "recall_feet"
            ],

        "runtime_sec":
            runtime,

    }


    ablation_results.append(
        result_row
    )


    # ========================================================
    # SAVE TRAINING HISTORY
    # ========================================================

    history_path = (
        ABLATION_HISTORY_ROOT
        /
        f"{experiment_name}_history.csv"
    )

    artifact[
        "history"
    ].to_csv(
        history_path,
        index=False,
    )


    # ========================================================
    # SAVE CHECKPOINT
    # ========================================================

    checkpoint_path = (
        ABLATION_CHECKPOINT_ROOT
        /
        f"{experiment_name}.pt"
    )


    torch.save(
        {

            "experiment":
                experiment_name,

            "target_subject":
                ABLATION_TARGET,

            "dann_max_lambda":
                max_lambda,

            "model_state_dict":
                {
                    key:
                        value.cpu()

                    for key,
                        value
                    in artifact[
                        "model"
                    ]
                    .state_dict()
                    .items()
                },

            "normalizer_median":
                artifact[
                    "normalizer"
                ].median_,

            "normalizer_iqr":
                artifact[
                    "normalizer"
                ].iqr_,

            "seed":
                ABLATION_SEED,

            "config":
                {

                    "epochs":
                        ABLATION_EPOCHS,

                    "batch_size":
                        ABLATION_BATCH_SIZE,

                    "learning_rate":
                        ABLATION_LR,

                    "weight_decay":
                        ABLATION_WEIGHT_DECAY,

                    "patience":
                        ABLATION_PATIENCE,

                    "dann_max_lambda":
                        max_lambda,

                    "architecture":
                        "USDA-Net",

                    "dataset":
                        DATASET,

                    "classes":
                        PRIMARY_CLASSES,

                    "channels":
                        FROZEN_COMMON_CHANNELS,

                    "sampling_rate":
                        TARGET_SFREQ,

                    "bandpass":
                        [
                            LOW_HZ,
                            HIGH_HZ,
                        ],

                    "epoch":
                        [
                            TMIN,
                            TMAX,
                        ],

                    "strict_loso":
                        True,
                },
        },

        checkpoint_path,
    )


    # ========================================================
    # STORE ARTIFACT
    # ========================================================

    ablation_artifacts[
        max_lambda
    ] = artifact


    # ========================================================
    # IMMEDIATE RESULT
    # ========================================================

    print(
        "\n"
        +
        "-" * 78
    )

    print(
        f"{experiment_name} COMPLETE"
    )

    print(
        "Best validation BAcc:",
        f"{artifact['best_val_balanced_accuracy'] * 100:.2f}%"
    )

    print(
        "Target accuracy:",
        f"{metrics['accuracy'] * 100:.2f}%"
    )

    print(
        "Target balanced accuracy:",
        f"{metrics['balanced_accuracy'] * 100:.2f}%"
    )

    print(
        "Target Macro-F1:",
        f"{metrics['macro_f1'] * 100:.2f}%"
    )

    print(
        "Target kappa:",
        f"{metrics['kappa']:.4f}"
    )

    print(
        "Runtime:",
        f"{runtime:.1f} sec"
    )

    print(
        "-" * 78
    )


    # ========================================================
    # MEMORY CLEANUP
    # ========================================================

    # Keep the artifacts dictionary so the three experiments
    # remain available for later analysis.

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# BUILD FINAL ABLATION TABLE
# ============================================================

ablation_results_df = (
    pd.DataFrame(
        ablation_results
    )
    .sort_values(
        "dann_max_lambda"
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# SAVE RESULTS
# ============================================================

ABLATION_RESULTS_CSV = (
    ABLATION_ROOT
    /
    "dann_ablation_results.csv"
)

ablation_results_df.to_csv(
    ABLATION_RESULTS_CSV,
    index=False,
)


# ============================================================
# FINAL DISPLAY
# ============================================================

print(
    "\n"
    +
    "=" * 78
)

print(
    "3-WAY DANN ABLATION COMPLETE"
)

print(
    "=" * 78
)

display(
    ablation_results_df
)

print(
    "\nSaved to:"
)

print(
    ABLATION_RESULTS_CSV
)

Ablation seed: 20260822

STARTING DANN_lambda_0.10
Target subject = S001
Maximum DANN λ = 0.10


/var/folders/kq/cbr0cvmd3sq82p6gbfd_31980000gn/T/ipykernel_2482/566835166.py:171: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


[DANN_lambda_0.10] [S001] ep=001 λmax=0.10 λ=0.000 lr=2.99e-04 task=1.0975 dom=4.6774 proto=1.1109 supcon=5.0668 valBAcc=33.33% valF1=16.67%
[DANN_lambda_0.10] [S001] ep=005 λmax=0.10 λ=0.000 lr=2.87e-04 task=0.9993 dom=4.5171 proto=0.9863 supcon=4.8807 valBAcc=41.58% valF1=37.85%
[DANN_lambda_0.10] [S001] ep=010 λmax=0.10 λ=0.010 lr=2.49e-04 task=0.8837 dom=4.2074 proto=0.8583 supcon=4.8321 valBAcc=37.88% valF1=25.73%
[DANN_lambda_0.10] [S001] ep=015 λmax=0.10 λ=0.052 lr=1.95e-04 task=0.7884 dom=4.7708 proto=0.7448 supcon=4.7805 valBAcc=42.63% valF1=35.62%
[DANN_lambda_0.10] [S001] ep=020 λmax=0.10 λ=0.078 lr=1.35e-04 task=0.7117 dom=4.4497 proto=0.6553 supcon=4.7112 valBAcc=38.41% valF1=31.57%
[DANN_lambda_0.10] [S001] ep=025 λmax=0.10 λ=0.091 lr=8.08e-05 task=0.5948 dom=4.4076 proto=0.5163 supcon=4.5901 valBAcc=43.11% valF1=40.85%
[DANN_lambda_0.10] Early stopping at epoch 28; best epoch=18; best val BAcc=45.86%

----------------------------------------------------------------------

/var/folders/kq/cbr0cvmd3sq82p6gbfd_31980000gn/T/ipykernel_2482/566835166.py:171: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


[DANN_lambda_0.20] [S001] ep=001 λmax=0.20 λ=0.000 lr=2.99e-04 task=1.0975 dom=4.6774 proto=1.1109 supcon=5.0668 valBAcc=33.33% valF1=16.67%


KeyboardInterrupt: 

In [ ]:
# CELL 25 — PER-SUBJECT PERFORMANCE PLOT

if len(results_df):
    plot_df = results_df.sort_values("accuracy")
    fig, ax = plt.subplots(figsize=(15,6))
    ax.bar(plot_df["target_subject"], plot_df["accuracy"]*100)
    ax.axhline(80, linestyle="--", linewidth=1.5, label="80% target")
    ax.set_xlabel("Held-out subject")
    ax.set_ylabel("Accuracy (%)")
    ax.set_title("USDA-Net — EEGMMIDB Strict LOSO Accuracy")
    ax.tick_params(axis="x", rotation=90)
    ax.legend()
    plt.tight_layout()
    fig.savefig(FIGURES_ROOT / "loso_accuracy_by_subject.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# CELL 26 — AGGREGATED PERFORMANCE DISTRIBUTIONS

fig, ax = plt.subplots(figsize=(10,6))
vals = [results_df["accuracy"].to_numpy()*100,
        results_df["balanced_accuracy"].to_numpy()*100,
        results_df["macro_f1"].to_numpy()*100]
ax.boxplot(vals, labels=["Accuracy","Balanced accuracy","Macro-F1"])
ax.axhline(80, linestyle="--", linewidth=1.2, label="80% target")
ax.set_ylabel("Score (%)")
ax.set_title("USDA-Net — Subject-Level LOSO Distribution")
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES_ROOT / "loso_metric_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

## Confusion matrix

For strict LOSO, the confusion matrix is aggregated over the predictions from all held-out subjects. The final matrix is meaningful only when each target fold has been evaluated with the target labels used **after** model selection.

In [ ]:
# CELL 27 — AGGREGATED CONFUSION MATRIX FROM RE-RUN PREDICTIONS
# This helper re-evaluates saved checkpoints fold-by-fold and constructs the global matrix.
# It does not fit anything and therefore does not affect the LOSO result.

# To keep the notebook resumable, only folds with saved checkpoints and saved normalizer stats are included.
# The checkpoint already stores the source-only normalization statistics used by that target fold.

all_true, all_pred = [], []

for target_subject in sorted(results_df.target_subject.astype(str)):
    ckpt_path = CHECKPOINT_ROOT / f"{target_subject}_USDA_Net.pt"
    if not ckpt_path.exists():
        continue
    ckpt = torch.load(ckpt_path, map_location="cpu")
    normalizer = SourceOnlyRobustNormalizer()
    normalizer.median_ = np.asarray(ckpt["normalizer_median"], dtype=np.float32)
    normalizer.iqr_ = np.asarray(ckpt["normalizer_iqr"], dtype=np.float32)
    target_idx = indices_for_subjects([target_subject])
    ds = EEGArrayDataset(ACTIVE_CACHE, target_idx, normalizer=normalizer, augment=False)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    n_domains = len(build_loso_fold(target_subject)[0])
    model = USDANet(n_channels=22, n_classes=3, n_domains=n_domains)
    model.load_state_dict(ckpt["model_state_dict"], strict=True)
    model = model.to(DEVICE)
    model.eval()
    with torch.no_grad():
        for xb,yb,_ in loader:
            xb=xb.to(DEVICE)
            pred=model(xb)["logits"].argmax(dim=1).cpu().numpy()
            all_pred.extend(pred.tolist())
            all_true.extend(yb.numpy().tolist())
    del model
    gc.collect()

if all_true:
    cm = confusion_matrix(all_true, all_pred, labels=[0,1,2])
    cm_df = pd.DataFrame(cm, index=PRIMARY_CLASSES, columns=PRIMARY_CLASSES)
    print(cm_df)
    fig, ax = plt.subplots(figsize=(6.5,5.5))
    im=ax.imshow(cm, interpolation="nearest")
    ax.set_title("USDA-Net — Aggregated Strict LOSO Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks(range(3), PRIMARY_CLASSES)
    ax.set_yticks(range(3), PRIMARY_CLASSES)
    for i in range(3):
        for j in range(3): ax.text(j,i,str(cm[i,j]),ha="center",va="center")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    fig.savefig(FIGURES_ROOT / "aggregated_confusion_matrix.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("No saved fold checkpoints available for confusion-matrix reconstruction.")

In [ ]:
# CELL 28 — RESEARCH MANIFEST / REPRODUCIBILITY RECORD

research_manifest = {
    "model": "USDA-Net",
    "dataset": "PhysioNet EEGMMIDB",
    "task": "3-class motor imagery",
    "classes": PRIMARY_CLASSES,
    "channel_count": N_CHANNELS,
    "channels": FROZEN_COMMON_CHANNELS,
    "sampling_rate_hz": TARGET_SFREQ,
    "bandpass_hz": [LOW_HZ, HIGH_HZ],
    "epoch_sec": [TMIN, TMAX],
    "epoch_shape": [N_CHANNELS, N_SAMPLES],
    "reference": "average reference over 22 common channels",
    "interpolation": False,
    "cache_normalized": False,
    "normalization": "source-training-subject-only robust median/IQR",
    "evaluation": "strict unseen-subject LOSO",
    "target_information_before_test": [],
    "loss": "cross_entropy + DANN + prototype + supervised_contrastive",
    "temporal_kernels": [3,7,15,31],
    "spectral_bands_initialization_hz": [[4,8],[8,12],[12,16],[16,22],[22,30],[30,38]],
    "embedding_dim": 128,
    "transformer_layers": 3,
    "transformer_heads": 4,
    "seed": SEED,
    "device": str(DEVICE),
    "cache": str(ACTIVE_CACHE),
    "results": str(LOSO_RESULTS_CSV),
}

manifest_path = MANIFEST_ROOT / "USDA_Net_EEGMMIDB_3class_research_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(research_manifest, f, indent=2)

print(json.dumps(research_manifest, indent=2))
print("\nResearch manifest saved to:", manifest_path)

## Interpretation rules for the paper

When reporting the final experiment:

- report **all EEGMMIDB subjects**, not just the easiest folds;
- report the mean ± SD across held-out subjects;
- include balanced accuracy and macro-F1 because class balance can vary after valid-epoch filtering;
- distinguish **strict LOSO** from any later unlabeled target-adaptation experiment;
- do not call the model “80% accurate” unless the completed LOSO mean actually reaches 80%;
- preserve the exact frozen preprocessing contract and source-only normalization in the Methods section.

### Planned ablation structure

The architecture should remain the proposed USDA-Net. Ablations remove one component at a time:

1. USDA-Net without DANN;
2. USDA-Net without prototype loss;
3. USDA-Net without supervised contrastive loss;
4. USDA-Net without learnable spectral branch;
5. USDA-Net without Transformer;
6. full USDA-Net.

These are **ablations of the same architecture**, not substitutions of the architecture with unrelated models.